# 03 — BERT Token Embedding Extraction

Runs `bert-base-uncased` (frozen, inference only) over every issue text and saves the token-level
embeddings per project, so the CNN training notebooks never need to touch BERT again.

**Input:** dataset `tawos-extracted` (all_issues_sp.csv from 01_extract_tawos).
**Output:**
- `emb/<PROJECT>.npy` — float16 array [n_issues, 128, 768], token embeddings (last hidden state)
- `meta/<PROJECT>.csv` — issue_id, issue_key, storypoint, n_tokens (real token count before padding)

Settings: max_length 128 (truncation for long descriptions), batch 64, fp16 autocast.
Same story point filter as the baseline: keep SP in (0, 40]. Needs GPU, internet on (model download).

In [ ]:
import glob, os, re
import numpy as np, pandas as pd

hits = glob.glob("/kaggle/input/**/all_issues_sp.csv", recursive=True)
assert hits, "attach the tawos-extracted dataset as Input"
df = pd.read_csv(hits[0])
df["storypoint"] = pd.to_numeric(df["storypoint"], errors="coerce")
df = df.dropna(subset=["storypoint"])
df = df[(df.storypoint > 0) & (df.storypoint <= 40)]
df["text"] = df["text"].fillna("").astype(str)
df = df.reset_index(drop=True)
print(len(df), "issues |", df.project.nunique(), "projects")
df[["project", "issue_key", "storypoint"]].head(3)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = "bert-base-uncased"
MAX_LEN = 128
BS = 64

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model = model.cuda().eval()
torch.set_grad_enabled(False)
print("gpu:", torch.cuda.get_device_name(0), "| params:", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")

In [ ]:
os.makedirs("/kaggle/working/emb", exist_ok=True)
os.makedirs("/kaggle/working/meta", exist_ok=True)

def safe(n):
    return re.sub(r"[^A-Za-z0-9_]", "_", str(n))

projects = sorted(df.project.unique())
summary = []
t_all = time.time() if False else None

import time
t0 = time.time()
for pi, proj in enumerate(projects):
    g = df[df.project == proj].reset_index(drop=True)
    out = np.empty((len(g), MAX_LEN, 768), dtype=np.float16)
    ntok = np.empty(len(g), dtype=np.int32)
    for b0 in range(0, len(g), BS):
        batch = g.text.values[b0:b0 + BS]
        enc = tok(list(batch), padding="max_length", truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            hs = model(**enc).last_hidden_state
        out[b0:b0 + len(batch)] = hs.half().cpu().numpy()
        ntok[b0:b0 + len(batch)] = enc["attention_mask"].sum(1).cpu().numpy()
    name = safe(proj)
    np.save(f"/kaggle/working/emb/{name}.npy", out)
    meta = g[["issue_id", "issue_key", "storypoint"]].copy()
    meta["n_tokens"] = ntok
    meta.to_csv(f"/kaggle/working/meta/{name}.csv", index=False)
    mb = out.nbytes / 1e6
    summary.append(dict(project=proj, n=len(g), size_mb=round(mb, 1),
                        trunc_pct=round(float((ntok >= MAX_LEN).mean() * 100), 1)))
    del out, ntok, enc, hs
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print(f"[{pi+1}/{len(projects)}] {proj}: {len(g)} issues, {mb:.0f} MB | {time.time()-t0:.0f}s elapsed", flush=True)

pd.DataFrame(summary).to_csv("/kaggle/working/extraction_summary.csv", index=False)
print("DONE in", round((time.time() - t0) / 60, 1), "min")

In [ ]:
# verification: every project file exists, shapes match meta, values sane
total = 0
for _, r in pd.read_csv("/kaggle/working/extraction_summary.csv").iterrows():
    name = safe(r.project)
    a = np.load(f"/kaggle/working/emb/{name}.npy", mmap_mode="r")
    m = pd.read_csv(f"/kaggle/working/meta/{name}.csv")
    assert a.shape == (len(m), MAX_LEN, 768), name
    total += a.nbytes
sample = np.load("/kaggle/working/emb/Sonatype_Nexus.npy", mmap_mode="r")[:8]
assert np.isfinite(np.asarray(sample)).all()
print("all 39 project files verified | total", round(total / 1e9, 2), "GB")

## Notes

- float16 halves the file size with no practical loss for a CNN input layer.
- `n_tokens` records the real token count; the fraction truncated at 128 is in `extraction_summary.csv`.
- In the CNN stage, load `emb/<project>.npy` with `mmap_mode="r"` and index by the row order of
  `meta/<project>.csv` — no need to hold everything in memory at once.